# Gold Layer — Order Status Funnel

Counts how many orders reached each stage of the lifecycle: purchased, approved, delivered to carrier, delivered to customer, plus the two drop-out categories (canceled, unavailable). Output is a single row with one count per stage.

## Design Decisions

* **Conditional aggregation with `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`** — produces multiple counts in one aggregation pass over `silver.orders`. The output is a wide single-row table rather than the more typical narrow one-row-per-status shape.
* **Funnel counts use timestamp columns, not status** — an order that reached `delivered_to_customer` also passed through `approved` and `delivered_to_carrier` earlier. Counting where the timestamp is non-null at each stage captures cumulative reach, which is what a funnel measures.
* **Canceled and unavailable counted by status** — these states don't have their own timestamps, so they're counted from the order_status column directly.
* **No status filter applied** — a funnel analysis should include all orders, including drop-outs, otherwise we can't measure the drop-off rate. This differs from the other Gold tables which exclude canceled/unavailable.
* **Source is `silver.orders` (98,059 rows), not Bronze** — the funnel is measured against validated orders. The 1,382 orders quarantined for event sequence violations are excluded from this baseline.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'
quarantine_schema = 'quarantine'

### Order Status by Funnel

In [0]:
df_orders_fun = spark.read.table(f'{catalog_name}.{silver_schema}.orders')
df_orders_fun.createOrReplaceTempView('orders_fun')


order_status_funnel_df = spark.sql('''Select count(*) as purchased,
SUM(CASE WHEN order_approved_at IS NOT NULL THEN 1 ELSE 0 END) as approved,
SUM(CASE WHEN order_delivered_carrier_date IS NOT NULL THEN 1 ELSE 0 END) as delivered_to_carrier,
SUM(CASE WHEN order_delivered_customer_date IS NOT NULL THEN 1 ELSE 0 END) as delivered_to_customer,
SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) as canceled,
SUM(CASE WHEN order_status = 'unavailable' THEN 1 ELSE 0 END) as unavailable
from orders_fun
''')

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8009868683507943>, line 1
----> 1 df_orders_fun = spark.read.table(f'{catalog_name}.{silver_schema}.orders')
      2 df_orders_fun.createOrReplaceTempView('orders_fun')

NameError: name 'catalog_name' is not defined

In [0]:
count_columns = ['purchased', 'approved', 'delivered_to_carrier', 'delivered_to_customer', 'canceled', 'unavailable']

check_not_null(order_status_funnel_df, count_columns)
check_value_range(order_status_funnel_df, [{'column': c, 'min_value': 0, 'inclusive': True} for c in count_columns])

In [0]:
order_status_funnel_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{gold_schema}.order_status_funnel')